# Costruire con i modelli della famiglia Meta 

## Introduzione 

Questa lezione coprirà: 

- Esplorare i due principali modelli della famiglia Meta - Llama 3.1 e Llama 3.2 
- Comprendere i casi d'uso e gli scenari per ogni modello 
- Esempio di codice per mostrare le caratteristiche uniche di ogni modello 


## La famiglia di modelli Meta 

In questa lezione esploreremo 2 modelli della famiglia Meta o "Llama Herd" - Llama 3.1 e Llama 3.2 

Questi modelli sono disponibili in diverse varianti e si trovano nel [catalogo Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst).

> **Nota:** GitHub Models sarà dismesso alla fine di luglio 2026. Ecco ulteriori dettagli sull'utilizzo di [Microsoft Foundry Models](https://learn.microsoft.com/en-us/azure/ai-foundry/model-inference/overview?WT.mc_id=academic-105485-koreyst) per prototipare con modelli di intelligenza artificiale.

Varianti del modello: 
- Llama 3.1 - 70B Instruct 
- Llama 3.1 - 405B Instruct 
- Llama 3.2 - 11B Vision Instruct 
- Llama 3.2 - 90B Vision Instruct 

*Nota: Llama 3 è disponibile anche in Microsoft Foundry Models ma non sarà trattato in questa lezione*



## Llama 3.1 

Con 405 miliardi di parametri, Llama 3.1 rientra nella categoria degli LLM open source. 

Il modello è un aggiornamento della versione precedente Llama 3 offrendo: 

- Finestra di contesto più ampia - 128k token contro 8k token 
- Numero massimo di token in output più alto - 4096 contro 2048 
- Migliore supporto multilingue - grazie all’aumento dei token di addestramento 

Queste caratteristiche permettono a Llama 3.1 di gestire casi d’uso più complessi nella creazione di applicazioni GenAI, tra cui: 
- Chiamata di funzione nativa - la capacità di chiamare strumenti esterni e funzioni al di fuori del flusso di lavoro LLM
- Migliore performance RAG - grazie alla finestra di contesto più ampia 
- Generazione di dati sintetici - la capacità di creare dati efficaci per compiti come il fine-tuning 



### Chiamata di Funzioni Native 

Llama 3.1 è stato messo a punto per essere più efficace nel fare chiamate a funzioni o strumenti. Ha anche due strumenti integrati che il modello può identificare come necessari da usare in base al prompt dell'utente. Questi strumenti sono: 

- **Brave Search** - Può essere usato per ottenere informazioni aggiornate come il meteo effettuando una ricerca sul web 
- **Wolfram Alpha** - Può essere usato per calcoli matematici più complessi, quindi non è necessario scrivere le proprie funzioni. 

Puoi anche creare i tuoi strumenti personalizzati che l'LLM può chiamare. 

Nell'esempio di codice qui sotto: 

- Definiamo gli strumenti disponibili (brave_search, wolfram_alpha) nel prompt di sistema. 
- Inviamo un prompt dell'utente che chiede del meteo in una certa città. 
- L'LLM risponderà con una chiamata allo strumento Brave Search che apparirà così `<|python_tag|>brave_search.call(query="Stockholm weather")` 

*Nota: Questo esempio fa solo la chiamata allo strumento; se desideri ottenere i risultati, dovrai creare un account gratuito sulla pagina API di Brave e definire la funzione stessa* 


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import AssistantMessage, SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Meta-Llama-3.1-405B-Instruct"

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


tool_prompt=f"""
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Environment: ipython
Tools: brave_search, wolfram_alpha
Cutting Knowledge Date: December 2023
Today Date: 23 July 2024

You are a helpful assistant<|eot_id|>
"""

messages = [
    SystemMessage(content=tool_prompt),
    UserMessage(content="What is the weather in Stockholm?"),

]

response = client.complete(messages=messages, model=model_name)

print(response.choices[0].message.content)






### Llama 3.2 

Nonostante sia un LLM, una limitazione di Llama 3.1 è la multimodalità. Cioè, la capacità di utilizzare diversi tipi di input come immagini come prompt e fornire risposte. Questa abilità è una delle caratteristiche principali di Llama 3.2. Queste caratteristiche includono anche: 

- Multimodalità - ha la capacità di valutare sia prompt testuali che immagini
- Variazioni da piccole a medie dimensioni (11B e 90B) - questo offre opzioni flessibili di distribuzione,
- Variazioni solo testuali (1B e 3B) - questo permette di distribuire il modello su dispositivi edge / mobili e fornisce bassa latenza

Il supporto multimodale rappresenta un grande passo nel mondo dei modelli open source. L'esempio di codice qui sotto prende sia un'immagine che un prompt testuale per ottenere un'analisi dell'immagine da Llama 3.2 90B.

### Supporto Multimodale con Llama 3.2


In [ ]:
%pip install azure-core
%pip install azure-ai-inference

In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import (
    SystemMessage,
    UserMessage,
    TextContentItem,
    ImageContentItem,
    ImageUrl,
    ImageDetailLevel,
)
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]
model_name = "Llama-3.2-90B-Vision-Instruct"


client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)

response = client.complete(
    messages=[
        SystemMessage(
            content="You are a helpful assistant that describes images in details."
        ),
        UserMessage(
            content=[
                TextContentItem(text="What's in this image?"),
                ImageContentItem(
                    image_url=ImageUrl.load(
                        image_file="sample.jpg",
                        image_format="jpg",
                        detail=ImageDetailLevel.LOW)
                ),
            ],
        ),
    ],
    model=model_name,
)

print(response.choices[0].message.content)


## L'apprendimento non si ferma qui, continua il Viaggio

Dopo aver completato questa lezione, dai un'occhiata alla nostra [collezione di apprendimento sull'IA Generativa](https://aka.ms/genai-collection?WT.mc_id=academic-105485-koreyst) per continuare a migliorare le tue conoscenze sull'IA Generativa!


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
